<a href="https://colab.research.google.com/github/DavidRuiz-beep/Proyecto/blob/main/clases/4_Aprendizaje_no_supervisado/2_Taller_Apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/4_Aprendizaje_no_supervisado/2_Taller_Apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# **Taller: Análisis de Patrones de Consumo Internacional con Apriori**

**IMPORTANTE**: Guarda una copia de este notebook en tu Google Drive o computador.

**Taller en grupos de 3**

**Nombres estudiantes:**

-
-
-

**Forma de entrega:**

- Nombrar el archivo de la siguiente forma:“Taller_Apriori_apellidos.ipynb”.
- Suba el Jupyter Notebook a su cuenta en Github y envíe el link en el siguiente Forms: https://forms.cloud.microsoft/r/qERdEpXpmx.

**IMPORTANTE:** No se recibirán talleres en Google Colab, el notebook debe estar subido en Github.

**Plazo de entrega:**

21 de abril de 2026, máximo a las 11:59 p.m. Tenga en cuenta que luego de esa hora el formulario en forms se cierra. El Jupupyter Notebook también debe quedar subido en Github antes de esa hora.

**Instrucciones Generales:**

Completa el código en las celdas marcadas con `### TU CÓDIGO AQUÍ ###`. Puedes añadir más celdas si lo requieres.

**Caso de Estudio: Consultoría para Global Retail Inc.**

**Contexto:** Una firma multinacional de e-commerce, "Global Retail Inc.", te ha contratado como consultor de datos. La empresa opera en múltiples países y ha notado que sus ventas y la efectividad de sus campañas de marketing varían significativamente entre regiones. Su hipótesis es que los patrones de compra y las asociaciones de productos son diferentes en cada mercado.

**Tu Misión:** Analizar el historial de transacciones de la empresa para descubrir y comparar las reglas de asociación de productos para dos de sus mercados más importantes en Latinoamérica: México y Colombia. Tu objetivo final es entregar recomendaciones de negocio accionables (ej. estrategias de cross-selling, promociones personalizadas) basadas en los patrones de consumo que descubras en cada país.

**Dataset:** Encuentra mayor información en el archivo "diccionario_alimentos_retail_top30.xlsx".

## Ejercicio 1: Configuración Inicial, Carga y Exploración de Datos

1.1 Importa las librerías necesarias

In [1]:
### TU CÓDIGO AQUÍ ###
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Configuraciones de visualización
pd.options.display.max_columns = None
pd.options.display.float_format = '{:,.2f}'.format

1.2 Carga el dataset "alimentos_retail_top30.csv" que se encuentra en el repositorio del curso, carpeta "datasets". El dataframe debe llamarse "df".

In [3]:
### TU CÓDIGO AQUÍ ###
df = pd.read_csv('alimentos_retail_top30.csv')

In [4]:
# Debe ser (6899, 8)
print("Dimensiones del DataFrame:")
print(df.shape)

Dimensiones del DataFrame:
(6899, 8)


In [5]:
print("\nInformación general del DataFrame:")
df.info()


Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6899 entries, 0 to 6898
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    6899 non-null   object 
 1   StockCode    6899 non-null   int64  
 2   Description  6899 non-null   object 
 3   Quantity     6899 non-null   int64  
 4   InvoiceDate  6899 non-null   object 
 5   UnitPrice    6899 non-null   float64
 6   CustomerID   6879 non-null   float64
 7   Country      6899 non-null   object 
dtypes: float64(2), int64(2), object(4)
memory usage: 431.3+ KB


1.3 Revisa si hay valores nulos en alguna columna y cuántos son

In [6]:
### TU CÓDIGO AQUÍ ###
print("Valores nulos por columna:")
print(df.isnull().sum())
print("\nPorcentaje de valores nulos:")
print((df.isnull().sum() / len(df)) * 100)

Valores nulos por columna:
InvoiceNo       0
StockCode       0
Description     0
Quantity        0
InvoiceDate     0
UnitPrice       0
CustomerID     20
Country         0
dtype: int64

Porcentaje de valores nulos:
InvoiceNo     0.00
StockCode     0.00
Description   0.00
Quantity      0.00
InvoiceDate   0.00
UnitPrice     0.00
CustomerID    0.29
Country       0.00
dtype: float64


1.4 Genera las estadísticas descriptivas de las variables numéricas

In [7]:
### TU CÓDIGO AQUÍ ###
print("Estadísticas descriptivas de variables numéricas:")
display(df.describe())

Estadísticas descriptivas de variables numéricas:


,StockCode,Quantity,UnitPrice,CustomerID
count,"6,899.00","6,899.00","6,899.00","6,879.00"
mean,"55,544.94",3.00,3.42,"15,024.12"
std,"25,875.73",1.43,1.06,"1,732.95"
min,"26,907.00",-5.00,1.65,"12,000.00"
25%,"31,048.00",2.00,2.36,"13,524.00"
50%,"42,889.00",3.00,3.39,"15,041.00"
75%,"87,297.00",4.00,4.44,"16,530.50"
max,"95,931.00",5.00,4.90,"17,999.00"


1.5 Observando las salidas del ejercicio anterior, ¿qué problemas potenciales identificas en las columnas CustomerID y Quantity?

Problemas identificados en CustomerID:
La columna CustomerID presenta 20 valores nulos (0.29% del dataset), lo que imposibilita asociar esas transacciones a clientes específicos, además, el tipo de dato es float64 cuando debería ser int64, ya que los identificadores de cliente son números enteros el rango de valores va de 12,000 a 17,999, lo cual parece coherente, pero es necesario verificar que no haya IDs fuera de este rango.

Problemas identificados en Quantity:
La columna Quantity muestra valores negativos (mínimo de -5), lo que representa devoluciones o cancelaciones de productos, estas transacciones no representan ventas reales y deben ser excluidas del análisis de patrones de compra, el rango principal es de 1 a 5 unidades con una media de 3, lo cual es normal para un retail de consumo diario.



## Ejercicio 2: Limpieza y Preprocesamiento de Datos

Los datos del mundo real rara vez son perfectos. Antes de cualquier análisis, debemos "sanear" nuestro dataset. Completa el código en cada paso según las instrucciones.

Crea un nuevo dataframe llamado "df_limpio" para los siguientes puntos.

2.1 **Manejo de Valores Nulos**: Las transacciones sin un CustomerID no son útiles para nosotros, ya que no podemos agrupar las compras de un cliente específico.

In [8]:
# TAREA: Elimina todas las filas donde 'CustomerID' es nulo.
### TU CÓDIGO AQUÍ ###
# Eliminar filas donde CustomerID es nulo
df_limpio = df.dropna(subset=['CustomerID'])

# Convertir CustomerID a entero
df_limpio['CustomerID'] = df_limpio['CustomerID'].astype(int)

print(f"Filas después de eliminar nulos: {df_limpio.shape[0]}")

Filas después de eliminar nulos: 6879


2.2 **Limpieza de Descripciones de Productos** Las descripciones pueden tener espacios en blanco al inicio o al final que podrían hacer que un mismo producto se cuente como dos diferentes.

In [9]:
### TU CÓDIGO AQUÍ ###
# Verificar cuántas descripciones únicas hay antes
print(f"Descripciones únicas antes: {df_limpio['Description'].nunique()}")

# Limpiar espacios en blanco
df_limpio['Description'] = df_limpio['Description'].str.strip()

# Verificar después de limpiar
print(f"Descripciones únicas después: {df_limpio['Description'].nunique()}")

Descripciones únicas antes: 25
Descripciones únicas después: 20


2.3 **Filtrado de Transacciones Anómalas**: Las facturas (InvoiceNo) que empiezan con 'C' indican una cancelación. Estas no son compras reales y deben ser eliminadas. Del mismo modo, las cantidades (Quantity) negativas representan devoluciones.

In [10]:
# TAREA: Elimina las filas que correspondan a cancelaciones.
### TU CÓDIGO AQUÍ ###
# Eliminar cancelaciones (InvoiceNo que empiezan con 'C')
df_limpio = df_limpio[~df_limpio['InvoiceNo'].astype(str).str.startswith('C')]

# Eliminar cantidades negativas
df_limpio = df_limpio[df_limpio['Quantity'] > 0]

# Verificar dimensiones
print(f"Dimensiones finales después de limpieza: {df_limpio.shape}")
# Debe ser aproximadamente (6864, 8)

Dimensiones finales después de limpieza: (6864, 8)


## Ejercicio 3: Análisis Comparativo por País

Ahora que los datos están limpios, vamos a segmentarlos y a aplicar el algoritmo Apriori para encontrar los patrones de compra en México y Colombia.

**Preparación de la Cesta de Mercado (Función)**

La siguiente función toma un dataframe, lo agrupa por factura y descripción, y lo transforma en el formato de matriz binaria que necesita el algoritmo Apriori. Estudia esta función, no necesitas modificarla.

In [11]:
def preparar_cesta(dataframe, pais):
    """Filtra por país y prepara la matriz de transacciones."""

    # Filtrar por el país de interés
    df_pais = dataframe[dataframe['Country'] == pais]

    # Crear la cesta: agrupar productos por factura
    cesta = (df_pais.groupby(['InvoiceNo', 'Description'])['Quantity']
             .sum().unstack().reset_index().fillna(0)
             .set_index('InvoiceNo'))

    # Convertir todas las cantidades positivas a 1 y todo lo demás a 0
    cesta_encoded = (cesta > 0).astype(int)

    return cesta_encoded

3.1 Análisis para México

In [31]:
### 3.1 Análisis para México ###

# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de México.
# Almacena el resultado en la variable cesta_mx.

### TU CÓDIGO AQUÍ ###
cesta_mx = preparar_cesta(df_limpio, 'México')

# Verificar que se creó correctamente
print(f"Matriz de transacciones para México:")
print(f"  - Transacciones: {cesta_mx.shape[0]}")
print(f"  - Productos únicos: {cesta_mx.shape[1]}")
print("\nPrimeras 5 transacciones (muestra):")
display(cesta_mx.head())

Filas para México: 3532
Facturas únicas en México: 1000
Productos únicos en México: 10
Dimensiones de la matriz: (1000, 10)
Transacciones: 1000
Productos únicos: 10
Matriz de transacciones para México:
  - Transacciones: 1000
  - Productos únicos: 10

Primeras 5 transacciones (muestra):


Description,AGUACATE,CEBOLLA,CHILE JALAPEÑO,CILANTRO,FRIJOL NEGRO,LIMÓN,QUESO FRESCO,TOMATE,TORTILLAS DE MAÍZ,TOTOPOS
InvoiceNo,,,,,,,,,,
537000,0,0,0,0,1,0,0,0,1,0
537001,0,1,1,1,0,0,0,1,0,0
537002,0,0,0,0,1,0,0,1,1,0
537003,0,1,1,1,0,0,0,1,0,0
537004,0,0,0,1,0,0,1,1,1,0


In [32]:
# TAREA: Aplica el algoritmo apriori para encontrar itemsets con un soporte mínimo de 2%.
# Almacena el resultado en la variable frequent_itemsets_mx.
# Muestra los 10 itemsets con el soporte más alto.

### TU CÓDIGO AQUÍ ###
from mlxtend.frequent_patterns import apriori

# Aplicar algoritmo Apriori con soporte del 2%
frequent_itemsets_mx = apriori(cesta_mx, min_support=0.02, use_colnames=True)

# Mostrar los 10 itemsets con mayor soporte
print("Top 10 itemsets más frecuentes en México:")
display(frequent_itemsets_mx.sort_values('support', ascending=False).head(10))


Top 10 itemsets más frecuentes en México:


,support,itemsets
2,0.42,(CHILE JALAPEÑO)
7,0.41,(TOMATE)
3,0.41,(CILANTRO)
1,0.41,(CEBOLLA)
8,0.38,(TORTILLAS DE MAÍZ)
4,0.36,(FRIJOL NEGRO)
0,0.35,(AGUACATE)
5,0.35,(LIMÓN)
9,0.33,(TOTOPOS)
31,0.33,"(CHILE JALAPEÑO, TOMATE)"


In [33]:
# TAREA: Genera las reglas de asociación. Queremos reglas con un Lift mayor a 2.
# Almacena el resultado en la variable rules_mx.

### TU CÓDIGO AQUÍ ###
from mlxtend.frequent_patterns import association_rules

# Generar reglas de asociación con lift > 2
rules_mx = association_rules(frequent_itemsets_mx, metric="lift", min_threshold=2)

print(f"Reglas encontradas con lift > 2: {len(rules_mx)}")

Reglas encontradas con lift > 2: 98


In [34]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas
# y las siguientes columnas: 'antecedents', 'consequents', 'antecedent support',
# 'consequent support', 'confidence', 'lift'

### TU CÓDIGO AQUÍ ###
# Ordenar por Lift y luego por Confianza (ambos de mayor a menor)
rules_mx_ordenadas = rules_mx.sort_values(['lift', 'confidence'], ascending=[False, False])

# Mostrar solo las columnas solicitadas
print("Top 10 reglas de asociación para México (ordenadas por Lift y Confianza):")
display(rules_mx_ordenadas[['antecedents', 'consequents', 'antecedent support',
                              'consequent support', 'confidence', 'lift']].head(10))

Top 10 reglas de asociación para México (ordenadas por Lift y Confianza):


,antecedents,consequents,antecedent support,consequent support,confidence,lift
91,"(CHILE JALAPEÑO, CEBOLLA)","(CILANTRO, TOMATE)",0.32,0.32,0.92,2.90
90,"(CILANTRO, TOMATE)","(CHILE JALAPEÑO, CEBOLLA)",0.32,0.32,0.93,2.90
89,"(CILANTRO, CEBOLLA)","(CHILE JALAPEÑO, TOMATE)",0.31,0.33,0.95,2.88
92,"(CHILE JALAPEÑO, TOMATE)","(CILANTRO, CEBOLLA)",0.33,0.31,0.90,2.88
12,"(LIMÓN, AGUACATE)",(TOTOPOS),0.27,0.33,0.93,2.82
13,(TOTOPOS),"(LIMÓN, AGUACATE)",0.33,0.27,0.75,2.82
88,"(CILANTRO, CHILE JALAPEÑO)","(TOMATE, CEBOLLA)",0.32,0.33,0.92,2.81
93,"(TOMATE, CEBOLLA)","(CILANTRO, CHILE JALAPEÑO)",0.33,0.32,0.91,2.81
74,"(TOMATE, LIMÓN, AGUACATE)",(TOTOPOS),0.02,0.33,0.91,2.76
75,(TOTOPOS),"(TOMATE, LIMÓN, AGUACATE)",0.33,0.02,0.06,2.76


3.3 Observa las 3 reglas con el Lift más alto para México (1, 3 y 5). **Interprétalas:** ¿Qué te dicen estas asociaciones? ¿Qué tipo de productos son?

3.4 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift

3.5 **Recomendación de Negocio:** Basado en estas reglas, ¿qué promoción o estrategia de venta específica podrías sugerir para el mercado mexicano?

3.6 Análisis para Colombia

In [ ]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de Colombia. Almacena el resultado en la variable cesta_co.
### TU CÓDIGO AQUÍ ###
cesta_co =

In [ ]:
# TAREA: Aplica el algoritmo apriori con un soporte mínimo del 2%.
# Almacena el resultado en la variable frequent_itemsets_co.
# Muestra los 10 itemsets con el soporte más alto.
### TU CÓDIGO AQUÍ ###
frequent_itemsets_co =


In [ ]:
# TAREA: Genera las reglas de asociación con un Lift mayor a 2. Almacena el resultado en la variable rules_co.
### TU CÓDIGO AQUÍ ###
rules_co =

In [ ]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
### TU CÓDIGO AQUÍ ###


3.7 Observa las 3 reglas con el Lift más alto para Colombia (1, 3 y 5). **Interprétalas:** ¿Qué patrones de consumo específicos del mercado colombiano revelan estas reglas? ¿Son diferentes a las de México?

3.8 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift

3.9 **Recomendación de Negocio:** ¿Qué campaña de marketing (diferente a la de México) podrías diseñar para los clientes colombianos?